In [5]:
from IPython.display import display

from config import SimConfig
from src.data_processing import load_and_cache_entire_fleet
from src.plants.hybrid_plant import FuelCellBatteryPlant
from src.utils.evaluation import VoyageBenchmarker
from src.plotting import plot_dashboard

In [ ]:
print("Initializing MARINER Validation Environment...")

# 1. Single unified configuration
config = SimConfig() 

# 2. Load the 1 Hz SOV dataset into RAM
fleet_cache = load_and_cache_entire_fleet(config)

# 3. Instantiate the benchmarking engine
benchmarker = VoyageBenchmarker(fleet_cache, exclude_days=[1, 2, 3])

# 4. The single physical truth: The Hybrid Plant
plant = FuelCellBatteryPlant(config)

Initializing MARINER Validation Environment...
Beginning memory staging of all 14 fleet files into RAM...
 -> Day 01 successfully cached in RAM.
 -> Day 02 successfully cached in RAM.
 -> Day 03 successfully cached in RAM.
 -> Day 04 successfully cached in RAM.
 -> Day 05 successfully cached in RAM.
 -> Day 06 successfully cached in RAM.
 -> Day 07 successfully cached in RAM.


In [ ]:
approaches = {
    # The Advanced Strategy (Actively manages battery SoC)
        "DP_Hybrid_Mean": {
            "strategy": "SDP",
            "sdp_variant": "MEAN_PROXY",
            "is_hybrid": True,   
            "config": config,
            "plant": plant
        },
    
    "DP_Hybrid_Tensor": {
        "strategy": "SDP",
        "sdp_variant": "TENSOR_SWEEP",
        "is_hybrid": True,   
        "config": config,
        "plant": plant
    },
    
    # The Wrapped Baseline DP (Blind to battery, optimizes H2 only)
    "DP_Baseline": {
        "strategy": "SDP",
        "is_hybrid": False,  # Triggers NaiveHybridWrapper 
        "config": config,
        "plant": plant
    },
    
    # The Wrapped Rule-Based Baseline (Moving threshold, blind to battery)
    "Heuristic_Baseline": {
        "strategy": "HEURISTIC",
        "is_hybrid": False,  # Triggers NaiveHybridWrapper 
        "config": config,
        "plant": plant
    }
}

In [ ]:
train_days=[4, 5, 6, 7, 8, 9, 10, 11, 12, 13] 
test_day=14

# Execute the simulations (All running inside the high-fidelity HybridSimulator)
df_results, sims = benchmarker.compare_approaches(approaches, train_days, test_day)

# Display tabular economics
print("\n--- Financial & Physical Summary ---")
display(df_results)

# Generate identical 4-panel dashboards for comparison
for approach_name, sim_instance in sims.items():
    plot_dashboard(
        sim=sim_instance, 
        approach_name=approach_name, 
        test_day=test_day, 
        layout='grid'
    )



Comparing 4 approaches | Train: [4, 5, 6, 7, 8, 9, 10, 11, 12, 13] | Test: Day 14
 -> Running: DP_Hybrid_Mean
 -> Launching MEAN_PROXY Solver...
 -> Running: DP_Hybrid_Tensor
 -> Launching TENSOR_SWEEP Solver...
 -> Triggering Offline MC Tensor Pre-Computation (50 paths)...
 -> Tensors Cached. Initiating Online Sweep...
 -> Running: DP_Baseline
 -> Running: Heuristic_Baseline

--- Financial & Physical Summary ---


,Total Cost ($),H2 Cost ($),FC Switch Cost ($),Bat. Degrade ($),Final SoC (%),Compute Time (s)
DP_Hybrid_Mean,5803.386901,5639.374309,18.75,145.262592,51.463589,88.393307
DP_Hybrid_Tensor,5795.409274,5605.060824,18.75,171.598451,50.857824,38.819553
DP_Baseline,5996.216223,5914.770674,37.50,43.945549,45.583155,0.096025
Heuristic_Baseline,6113.497927,5377.182101,675.00,61.315826,37.518689,0.039336


In [ ]:
# Run Chronological Forward Chaining for the Hybrid Mean SDP
# print("--- APPROACH B: CHRONOLOGICAL FORWARD CHAINING ---")
# df_forward = benchmarker.run_forward_chaining(hybrid_mean_sdp, min_train_days=1)
# display(df_forward)

# Run Leave-One-Out for the Hybrid Mean SDP
print("\n--- APPROACH A: LEAVE ONE OUT ---")
df_loo_mean = benchmarker.run_leave_one_out(approaches["DP_Hybrid_Mean"])
display(df_loo_mean)

In [ ]:
# Run Leave-One-Out for the Hybrid Tensor SDP
print("\n--- APPROACH A: LEAVE ONE OUT ---")
df_loo_tensor = benchmarker.run_leave_one_out(approaches["DP_Hybrid_Tensor"])
display(df_loo_tensor)